In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.tft import TemporalFusionTransformer

In [3]:
train_loader, valid_loader, test_loader, scaler = preprocess('ACB', 'tft', verbose=True)

Train shape: torch.Size([1094, 30, 4]), torch.Size([1094, 4])
Valid shape: torch.Size([121, 30, 4]), torch.Size([121, 4])
Test shape: torch.Size([328, 30, 4]), torch.Size([328, 4])


In [ ]:
model = TemporalFusionTransformer()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)
criterion = nn.SmoothL1Loss()

In [ ]:
best_val_loss = float('inf')
n_epochs = 50

for epoch in range(1, n_epochs + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch, y_batch
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch, y_batch
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
    val_loss /= len(valid_loader.dataset)

    scheduler.step()

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'checkpoints_tft/tft_ACB.pth')

    if epoch % 10 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch:3d}/{n_epochs}: "
              f"Train Loss = {train_loss:.6f}, "
              f"Valid Loss = {val_loss:.6f}, "
              f"Best Val Loss = {best_val_loss:.6f}, "
			  f"LR = {optimizer.param_groups[0]['lr']:.6f}")

Epoch  10/50: Train Loss = 0.001261, Valid Loss = 0.000797, Best Val Loss = 0.000797, LR = 0.000800
Epoch  20/50: Train Loss = 0.001095, Valid Loss = 0.000780, Best Val Loss = 0.000674, LR = 0.000640
Epoch  30/50: Train Loss = 0.001057, Valid Loss = 0.000679, Best Val Loss = 0.000665, LR = 0.000512
Epoch  40/50: Train Loss = 0.001050, Valid Loss = 0.000677, Best Val Loss = 0.000660, LR = 0.000410
Epoch  50/50: Train Loss = 0.001060, Valid Loss = 0.000754, Best Val Loss = 0.000660, LR = 0.000328


In [9]:
def eval(symbol):
    _, _, test_loader, scaler = preprocess(symbol, 'tft')
    model.load_state_dict(torch.load(f'checkpoints_tft/tft_{symbol}.pth', map_location='cpu'))
    model.eval()

    # Thu thập dự đoán và nhãn
    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch
            preds = model(X_batch).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_preds   = np.vstack(all_preds)   # (n_samples, 5)
    all_targets = np.vstack(all_targets)

    # Inverse scaling
    all_preds_inv   = scaler.inverse_transform(all_preds)
    all_targets_inv = scaler.inverse_transform(all_targets)

    # Tính metrics
    r2   = r2_score(all_targets_inv, all_preds_inv, multioutput='uniform_average')
    mape = mean_absolute_percentage_error(all_targets_inv, all_preds_inv) * 100
    
    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")
    return r2, mape

In [10]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    r2, mape = eval(symbol)
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R^2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R^2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R^2: 0.9485, MAPE: 0.8066
Symbol: BCM, R^2: 0.9666, MAPE: 1.0875
Symbol: BID, R^2: 0.9309, MAPE: 0.8695
Symbol: BVH, R^2: 0.9834, MAPE: 0.8777
Symbol: CTG, R^2: 0.9707, MAPE: 1.0293
Symbol: FPT, R^2: 0.9836, MAPE: 1.6862
Symbol: GAS, R^2: 0.9652, MAPE: 0.6509
Symbol: GVR, R^2: 0.9771, MAPE: 1.3694
Symbol: HDB, R^2: 0.9818, MAPE: 0.9128
Symbol: HPG, R^2: 0.9265, MAPE: 0.8565
Symbol: LPB, R^2: 0.9926, MAPE: 1.5940
Symbol: MBB, R^2: 0.9659, MAPE: 0.9177
Symbol: MSN, R^2: 0.9664, MAPE: 0.9353
Symbol: MWG, R^2: 0.9870, MAPE: 1.0034
Symbol: PLX, R^2: 0.9856, MAPE: 0.8968
Symbol: SAB, R^2: 0.8828, MAPE: 1.4555
Symbol: SHB, R^2: 0.9720, MAPE: 0.7847
Symbol: SSB, R^2: 0.9626, MAPE: 0.9236
Symbol: SSI, R^2: 0.9509, MAPE: 0.9509
Symbol: STB, R^2: 0.9838, MAPE: 0.9218
Symbol: TCB, R^2: 0.9849, MAPE: 0.8908
Symbol: TPB, R^2: 0.9634, MAPE: 0.9207
Symbol: VCB, R^2: 0.9079, MAPE: 0.6263
Symbol: VHM, R^2: 0.9721, MAPE: 1.1631
Symbol: VIB, R^2: 0.9504, MAPE: 0.7624
Symbol: VIC, R^2: 0.9800,